# Project 02: Train a Loan Approval Classifier (ML and DL Intro)

Train three models on one loan dataset and compare how well each predicts loan approval on data it has not seen.

Run top to bottom. Each step matches the README.

In [ ]:
# Step 0: find the project data
import os
PROJECT = "projects/02-ml-and-dl-intro"

# Find loan_data.csv: next to this notebook, or inside the cloned repo.
for path in ["loan_data.csv", os.path.join(PROJECT, "loan_data.csv")]:
    if os.path.exists(path):
        os.chdir(os.path.dirname(os.path.abspath(path)))
        break

# If the file is still missing, upload it (it goes next to this notebook).
if not os.path.exists("loan_data.csv"):
    try:
        from google.colab import files
        print("loan_data.csv not found. Upload it from the 02-ml-and-dl-intro folder.")
        files.upload()
    except ImportError:
        print("Not running in Colab. Put loan_data.csv in the same folder as this notebook.")
print("loan_data.csv found:", os.path.exists("loan_data.csv"))

## Step 1: install the pinned packages

Colab already has scikit-learn, pandas and numpy. This cell installs the exact versions from `requirements.txt`. A restart prompt may appear. If it does, restart the runtime and run the cells again from Step 0.

In [ ]:
%pip install -q numpy==2.1.3 pandas==2.2.3 scikit-learn==1.5.2

## Step 2: API key

This project needs **no API key**. Skip this step.

### Step 1: Load the data
**What:** Read `loan_data.csv` into a table (a DataFrame). Each row is one applicant.

**Why:** `Loan_Status` is the answer we want to predict. Data that includes the answer is called **labeled data**, and it is what supervised learning needs.

In [ ]:
import pandas as pd
data = pd.read_csv("loan_data.csv")
print(data.head())

### Step 2: Prepare the columns
**What:** Divide the income and loan columns by 1000, change `Education` and `Loan_Status` from text to 0/1, and one-hot encode `Property_Area` (one 0/1 column per area).

**Why:** Models only work with numbers. Smaller numbers help training run smoothly. This step is the "preprocess data" step of the five-step workflow.

**Run this cell once per session.** It changes `data` in place, so running it again divides the income columns again. If you do, use Runtime, Restart session, then run from Step 0.

In [ ]:
for col in ["Applicant_Income", "Coapplicant_Income", "Loan_Amount"]:
    data[col] = data[col] / 1000
data["Education"] = data["Education"].map({"Not Graduate": 0, "Graduate": 1})
data["Loan_Status"] = data["Loan_Status"].map({"N": 0, "Y": 1})
data = pd.get_dummies(data, columns=["Property_Area"], dtype=int)

### Step 3: Split into inputs (X), target (y), and train/test sets
**What:** Put every column except `Loan_Status` into `X` (the inputs). Put `Loan_Status` into `y` (the target). Then split the rows: 80% for training and 20% for testing.

**Why:** The model learns from the training rows. The test rows are held back, so we can check the model on applicants it has never seen. Without this split, a score only shows how well the model memorized the data.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=["Loan_Status"])
y = data["Loan_Status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training rows: {len(X_train)}   Testing rows: {len(X_test)}")

### Step 4: Train logistic regression
**What:** Fit a logistic regression model and measure accuracy on both sets. Accuracy is the share of correct predictions.

**Why:** Logistic regression is a simple, standard classifier. Its output is a yes or no answer (1 or 0), which fits a loan decision.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
print(logreg.score(X_train, y_train))
print(logreg.score(X_test, y_test))

### Step 5: Show overfitting with a decision tree
**What:** Train a decision tree with no depth limit, then compare its train and test accuracy.

**Why:** With no limit, the tree keeps splitting until it memorizes the training rows. It scores almost perfectly on data it has already seen, but worse on new data. The gap between the two scores is **overfitting**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)  # no max_depth = no limit
tree.fit(X_train, y_train)
print(tree.score(X_train, y_train))  # close to 1.0
print(tree.score(X_test, y_test))    # lower

### Step 6: Train a small neural network
**What:** Train a multi-layer perceptron (MLP) with one hidden layer of 8 neurons. Its hidden layer uses the ReLU activation function.

**Why:** This is the simplest neural network. During training, each **epoch** runs forward propagation (inputs flow forward to a prediction) and backward propagation (the error flows back and the weights are adjusted). ReLU is applied to each hidden neuron's output so the network can learn non-linear patterns.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(8,), activation="relu",
                    max_iter=2000, random_state=42)
mlp.fit(X_train, y_train)
print(mlp.score(X_test, y_test))

### Step 7: Compare the three models
**What:** Print the test accuracy of each model side by side.

**Why:** Test accuracy is the fair comparison, because every model is scored on the same unseen rows. A simple model can beat a complex one on a small dataset.

In [ ]:
print("--- Summary (test accuracy) ---")
print(f"Logistic regression: {logreg.score(X_test, y_test):.3f}")
print(f"Decision tree:       {tree.score(X_test, y_test):.3f}")
print(f"Neural network:      {mlp.score(X_test, y_test):.3f}")

## Checkpoints and what you learned

- **After Step 1:** first 5 rows with the columns `Applicant_Income` through `Loan_Status`
- **After Step 3:** `Training rows: 240` and `Testing rows: 60` (the 300 rows split 80/20)
- **After Step 4:** two accuracies, both between 0.70 and 0.85
- **After Step 5:** train accuracy of 1.000 and a lower test accuracy. The gap is the overfitting
- **After Step 6:** one test accuracy for the neural network, close to the other two models
- **After Step 7:** a summary of three test accuracies

Your numbers should match these, because the code uses a fixed random seed (`42`). If you change the seed or the split, your numbers will change. That is normal.

What you learned:
- Labeled data has inputs (X) and a known target (y). Supervised learning uses it.
- A train/test split checks a model on data it has not seen.
- Logistic regression is a simple classifier and a strong baseline.
- Overfitting is a large gap between train accuracy and test accuracy.
- A neural network stacks layers of weighted connections. Forward propagation makes predictions, backward propagation adjusts the weights, and activation functions add non-linear behavior.

Run locally instead? See the README setup section.